# Retention Analysis — Tidemill API vs Stripe

Cohort retention analysis comparing the Tidemill engine with Stripe subscription lifecycle data.

**API base:** `http://localhost:8000` (after `make check-e2e`) or set `TIDEMILL_API` env var

In [ ]:
import os
import warnings
from datetime import UTC, datetime

import matplotlib.pyplot as plt
import pandas as pd
import requests
import seaborn as sns
import stripe

warnings.filterwarnings("ignore", "Unverified HTTPS")

API = os.environ.get("TIDEMILL_API", "http://localhost:8000")
stripe.api_key = os.environ["STRIPE_API_KEY"]

START, END = "2025-09-01", "2026-03-31"
plt.rcParams.update({"figure.figsize": (12, 6)})


def api_get(path, **params):
    r = requests.get(f"{API}{path}", params=params)
    r.raise_for_status()
    return r.json()

## 1. Cohort retention from Tidemill API

In [ ]:
retention = api_get("/api/metrics/retention", start=START, end=END)
df_ret = pd.DataFrame(retention)
print(f"Retention data points: {len(df_ret)}")
df_ret

## 2. Build cohort retention from Stripe (ground truth)

Assign each customer to the month they first subscribed, then check which months they remained active.

In [ ]:
# Collect all subscriptions from Stripe across test clocks
clock_ids = [c.id for c in stripe.test_helpers.TestClock.list(limit=100).auto_paging_iter()]
all_subs = []
for cid in clock_ids:
    for sub in stripe.Subscription.list(
        limit=100, test_clock=cid, status="all"
    ).auto_paging_iter():
        all_subs.append(dict(sub))

# Build customer lifecycle: first subscription date, cancel date
customers = {}
for s in all_subs:
    cust_id = s["customer"]
    created = datetime.fromtimestamp(s["created"], tz=UTC)
    canceled_at = (
        datetime.fromtimestamp(s["canceled_at"], tz=UTC) if s.get("canceled_at") else None
    )

    if cust_id not in customers:
        customers[cust_id] = {"first_sub": created, "canceled_at": None, "status": s["status"]}

    if created < customers[cust_id]["first_sub"]:
        customers[cust_id]["first_sub"] = created

    if s["status"] == "canceled" and canceled_at:
        customers[cust_id]["canceled_at"] = canceled_at
        customers[cust_id]["status"] = "canceled"

df_cust = pd.DataFrame.from_dict(customers, orient="index")
df_cust["cohort_month"] = df_cust.first_sub.dt.to_period("M")
df_cust.index.name = "customer_id"

# Generate month range
months = pd.period_range(START, END, freq="M")

# For each customer, determine if they were active in each month
records = []
for cust_id, row in df_cust.iterrows():
    cohort = row.cohort_month
    for month in months:
        if month < cohort:
            continue
        # Customer is active if not canceled, or canceled after this month
        active = True
        if row.canceled_at:
            cancel_month = row.canceled_at.to_period("M")
            if month >= cancel_month:
                active = False
        records.append(
            {
                "customer_id": cust_id,
                "cohort_month": str(cohort),
                "active_month": str(month),
                "months_since": (month - cohort).n,
                "active": active,
            }
        )

df_activity = pd.DataFrame(records)
print(f"Customers: {len(df_cust)}")
print(f"Cohort months: {df_cust.cohort_month.unique()}")
print(f"Activity records: {len(df_activity)}")

## 3. Cohort retention heatmap (Stripe)

In [ ]:
# Build cohort retention matrix
cohort_sizes = df_activity.groupby("cohort_month").customer_id.nunique()
retention_counts = (
    df_activity[df_activity.active]
    .groupby(["cohort_month", "months_since"])
    .customer_id.nunique()
    .unstack(fill_value=0)
)

retention_pct = retention_counts.div(cohort_sizes, axis=0) * 100

# Heatmap
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(
    retention_pct,
    annot=True,
    fmt=".0f",
    cmap="RdYlGn",
    vmin=0,
    vmax=100,
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "Retention %"},
)
ax.set_title("Cohort Retention Heatmap (Stripe ground truth)")
ax.set_xlabel("Months since signup")
ax.set_ylabel("Cohort month")
plt.tight_layout()
plt.show()

print("\nCohort sizes:")
for cohort, size in cohort_sizes.items():
    print(f"  {cohort}: {size} customers")

## 4. Compare Tidemill retention with Stripe

In [ ]:
# Compare Tidemill retention data with Stripe ground truth
if len(df_ret) > 0:
    tidemill_retention = df_ret.copy()
    tidemill_retention["retention_pct"] = (
        tidemill_retention["active_count"] / tidemill_retention["cohort_size"] * 100
    )

    # Merge with Stripe data for comparison
    stripe_flat = (
        df_activity[df_activity.active]
        .groupby(["cohort_month", "active_month"])
        .customer_id.nunique()
        .reset_index()
        .rename(columns={"customer_id": "stripe_active"})
    )
    stripe_flat["stripe_cohort_size"] = stripe_flat.cohort_month.map(cohort_sizes)
    stripe_flat["stripe_retention_pct"] = (
        stripe_flat.stripe_active / stripe_flat.stripe_cohort_size * 100
    )

    compare = tidemill_retention.merge(
        stripe_flat, on=["cohort_month", "active_month"], how="outer"
    ).fillna(0)

    print("Tidemill vs Stripe retention comparison:")
    print(
        compare[
            [
                "cohort_month",
                "active_month",
                "cohort_size",
                "active_count",
                "retention_pct",
                "stripe_active",
                "stripe_retention_pct",
            ]
        ].to_string(index=False)
    )
else:
    print("No retention data from Tidemill API yet.")
    print("This is expected — retention requires cohort assignment during event processing.")
    print()
    print("Showing Stripe-only retention analysis above.")

## 5. Retention curve

In [ ]:
# Average retention curve across all cohorts
avg_retention = retention_pct.mean(axis=0)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(
    avg_retention.index, avg_retention.values, "o-", color="#3498db", linewidth=2, markersize=8
)
ax.fill_between(avg_retention.index, avg_retention.values, alpha=0.1, color="#3498db")
ax.set_title("Average Retention Curve (Stripe)")
ax.set_xlabel("Months since signup")
ax.set_ylabel("Retention %")
ax.set_ylim(0, 105)
ax.set_xticks(avg_retention.index)
ax.grid(True, alpha=0.3)

for x, y in zip(avg_retention.index, avg_retention.values, strict=True):
    ax.annotate(
        f"{y:.0f}%",
        (x, y),
        textcoords="offset points",
        xytext=(0, 12),
        ha="center",
        fontweight="bold",
    )

plt.tight_layout()
plt.show()

print(f"\nMonth 0 retention: {avg_retention.iloc[0]:.0f}%")
if len(avg_retention) > 1:
    print(f"Month 1 retention: {avg_retention.iloc[1]:.0f}%")
if len(avg_retention) > 5:
    print(f"Month 6 retention: {avg_retention.iloc[-1]:.0f}%")

## 6. Monthly NRR and GRR

Net Revenue Retention and Gross Revenue Retention per month from the Tidemill API.

- **NRR** = (start MRR + expansion - contraction - churn) / start MRR
- **GRR** = (start MRR - contraction - churn) / start MRR

NRR > 100% means expansion outpaces churn. GRR is always <= 100%.

In [ ]:
# Monthly NRR and GRR from Tidemill API
months = pd.date_range(START, END, freq="MS")
monthly_rr = []
for i in range(len(months) - 1):
    s = months[i].strftime("%Y-%m-%d")
    e = months[i + 1].strftime("%Y-%m-%d")
    nrr = api_get("/api/metrics/retention", start=s, end=e, query_type="nrr")
    grr = api_get("/api/metrics/retention", start=s, end=e, query_type="grr")
    monthly_rr.append(
        {
            "month": months[i].strftime("%Y-%m"),
            "nrr": nrr,
            "grr": grr,
        }
    )

df_rr = pd.DataFrame(monthly_rr)
print("Monthly Revenue Retention:")
for _, row in df_rr.iterrows():
    nrr_s = f"{row.nrr * 100:.1f}%" if row.nrr is not None else "N/A"
    grr_s = f"{row.grr * 100:.1f}%" if row.grr is not None else "N/A"
    print(f"  {row.month}:  NRR={nrr_s:>8s}   GRR={grr_s:>8s}")

In [ ]:
# Plot NRR and GRR
fig, ax = plt.subplots(figsize=(12, 5))

nrr_vals = [r * 100 if r is not None else float("nan") for r in df_rr.nrr]
grr_vals = [r * 100 if r is not None else float("nan") for r in df_rr.grr]

x = range(len(df_rr))
ax.plot(x, nrr_vals, "o-", color="#3498db", linewidth=2.5, markersize=8, label="NRR")
ax.plot(x, grr_vals, "s--", color="#2ecc71", linewidth=2.5, markersize=8, label="GRR")
ax.axhline(100, color="#95a5a6", linewidth=1, linestyle=":", label="100% baseline")

for i, (nv, gv) in enumerate(zip(nrr_vals, grr_vals, strict=True)):
    if not pd.isna(nv):
        ax.annotate(
            f"{nv:.0f}%",
            (i, nv),
            textcoords="offset points",
            xytext=(0, 12),
            ha="center",
            fontweight="bold",
            fontsize=9,
            color="#3498db",
        )
    if not pd.isna(gv):
        ax.annotate(
            f"{gv:.0f}%",
            (i, gv),
            textcoords="offset points",
            xytext=(0, -16),
            ha="center",
            fontweight="bold",
            fontsize=9,
            color="#2ecc71",
        )

ax.set_xticks(x)
ax.set_xticklabels(df_rr.month, rotation=45)
ax.set_title("Monthly Revenue Retention")
ax.set_ylabel("Retention (%)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()